# Topic interpretation of the original abstract clusters

Restored to the original **3,819 abstracts** and their existing full BGE-M3 embeddings. This notebook loads the original saved six-cluster K-means and Leiden partitions without fitting, preserving the source memberships and matching topic labels. No subject filter is active.

Use **Python (batill)** and restart the kernel before running. Results from the filtered sample have been cleared from this notebook; its separate files remain available as an archive.


## 1. Validate inputs and load existing assignments

The audited abstract loader checks the source CSV, embedding run, Scopus IDs, saved text and metadata, token audit, matrix shape and normalization. The partition loader checks the export's input and assignment checksums, original cluster IDs and actual exported cluster counts. Assignments are aligned by Scopus EID, not by coincidental row position. Missing or stale exports stop the notebook; there is no reclustering fallback.


In [ ]:
from pathlib import Path
import sys
from datetime import datetime, timezone
from importlib.metadata import version

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'src' / 'batill').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from batill.abstract_topic_analysis import (
    load_abstract_topic_corpus, load_abstract_topic_partitions, abstract_partition_input_id,
    describe_abstract_partition, write_abstract_review_cards,
)
from batill.topic_analysis import build_vocabulary, topic_terms, partition_agreement, STOP_WORDS
from batill.storage import file_hash, write_json

from batill.abstract_scope import resolve_abstract_inputs, abstract_partition_folder
ABSTRACT_RUN_DIR, ABSTRACT_SOURCE = resolve_abstract_inputs(
    ROOT, use_original=True)
print('Abstract corpus:', ABSTRACT_SOURCE)
papers, embeddings, provenance = load_abstract_topic_corpus(ABSTRACT_RUN_DIR, ABSTRACT_SOURCE)
INPUT_ID = abstract_partition_input_id(papers.paper_id, embeddings)
PARTITION_DIR = ROOT / 'outputs' / 'abstract_text_clusters' / INPUT_ID[:16] / abstract_partition_folder(provenance)
solutions, partition_manifest = load_abstract_topic_partitions(
    PARTITION_DIR, papers, embeddings, provenance=provenance,
)
RESULTS = (ROOT / 'reports' / 'topic_analysis_abstracts' / INPUT_ID[:16]
           / partition_manifest['assignments_sha256'][:16])
RESULTS.mkdir(parents=True, exist_ok=True)
metadata_columns = ['paper_id', 'title', 'doi', 'year', 'journal', 'document_type',
                    'document_language', 'abstract_word_count']
papers[metadata_columns].to_csv(RESULTS / 'papers.csv', index=False)
write_json(RESULTS / 'source_partition_manifest.json', partition_manifest)
print(f"Abstracts: {len(papers):,}; source: {partition_manifest['source_notebook']}")
print('Exact assignment snapshot:', PARTITION_DIR)
print('Evidence folder:', RESULTS)
display(pd.DataFrame([
    {'solution': name, 'method': fit['method'], 'source_variable': fit['source_variable'],
     'clusters': len(np.unique(fit['labels'])), 'abstracts': len(fit['labels']),
     'selected_seed': fit['seed'], 'resolution': fit.get('resolution'),
     'cosine_silhouette': fit['silhouettes'].mean()}
    for name, fit in solutions.items()
]))


## 2. Extract topic evidence for these memberships

One fixed vocabulary uses the exact embedded **title plus cleaned abstract** for each paper. It contains words and contiguous two-word phrases appearing in at least three records and at most 90% of records. English stopwords and formatting/URL tokens are removed, with no stemming or synonym merging. Bigrams do not bridge punctuation, stopwords or the title/abstract separator.

**c-TF-IDF** pools counts within each group and weights terms by corpus frequency. **Per-paper TF-IDF contrast** subtracts the mean normalized term score outside a group from the mean inside it. A ranked term must occur in at least two records and 10% of the group. The top lists can be shorter than 20 terms because abstracts are short. Inside/outside coverage counts support interpretation; these scores are not significance tests.

Each cluster gets three central papers (highest mean within-group cosine similarity), two lowest-silhouette papers and up to two additional seeded random papers. Excerpts of up to 600 characters come from the cleaned abstract, with exact character offsets; the full abstract, title, DOI, year and journal are also saved. These are source passages, not generated summaries.


In [ ]:
vocabulary = build_vocabulary(papers, min_df=3, max_df=.9)
write_json(RESULTS / 'vocabulary.json', vocabulary['terms'].tolist())
write_json(RESULTS / 'stopwords.json', sorted(STOP_WORDS))
profiles, memberships, examples, rankings = [], [], [], []
for name, fit in solutions.items():
    terms = topic_terms(vocabulary, fit['labels'], top_n=20)
    summary, membership, sample = describe_abstract_partition(papers, embeddings, fit, terms)
    for frame, container in [(summary, profiles), (membership, memberships),
                             (sample, examples), (terms, rankings)]:
        frame.insert(0, 'solution', name)
        container.append(frame)
profiles = pd.concat(profiles, ignore_index=True)
assignments = pd.concat(memberships, ignore_index=True)
review_papers = pd.concat(examples, ignore_index=True)
term_scores = pd.concat(rankings, ignore_index=True)
for frame, filename in [(profiles, 'cluster_profiles'), (assignments, 'memberships'),
                        (review_papers, 'review_papers'), (term_scores, 'term_scores')]:
    frame.to_csv(RESULTS / f'{filename}.csv', index=False)
# Explicit equality checks after all interpretation work.
for name, fit in solutions.items():
    exported_labels = assignments.loc[assignments.solution.eq(name), 'cluster'].to_numpy()
    np.testing.assert_array_equal(exported_labels, fit['labels'])
print('Memberships preserved exactly. Vocabulary terms:', len(vocabulary['terms']))
display(profiles[['solution', 'cluster', 'papers', 'mean_silhouette', 'contrast_terms']])

## 3. Names for each method's selected clusters

Names are **AI-assisted analyst proposals**, based on the ranked terms, document coverage, titles and sampled abstracts. They are interpretive inputs, not outputs of an automatic naming algorithm and not yet user-validated labels. The same proposal file reproduces the names; the code independently recomputes their supporting evidence.

Each name is bound to its method, original cluster ID and a checksum of the member IDs and exact source texts. `configs/abstract_topic_labels.csv` records naming rationale, caveats and blank reviewer fields and is never overwritten here. Missing or changed proposals display **Name pending review**. The two methods are named separately, without importing names from the PDF corpus.


In [ ]:
PROPOSAL_FILE = ROOT / 'configs' / 'abstract_topic_labels.csv'
if PROPOSAL_FILE.exists():
    proposals = pd.read_csv(PROPOSAL_FILE, keep_default_na=False)
else:
    proposals = pd.DataFrame(columns=['solution', 'cluster', 'membership_id',
                                      'proposed_label', 'status'])
named_profiles = write_abstract_review_cards(
    profiles, term_scores, review_papers, proposals,
    RESULTS / 'cluster_review_cards.md',
)
named_profiles['display_cluster'] = [
    f'L{cluster}' if solution.startswith('embedding_leiden_') else str(cluster)
    for solution, cluster in zip(named_profiles.solution, named_profiles.cluster)
]
named_profiles.to_csv(RESULTS / 'named_cluster_profiles.csv', index=False)
named_assignments = assignments.merge(
    named_profiles[['solution', 'cluster', 'proposed_label', 'proposal_matched']],
    on=['solution', 'cluster'], how='left', validate='many_to_one',
)
named_assignments.to_csv(RESULTS / 'named_memberships.csv', index=False)
for name, fit in solutions.items():
    display(Markdown(f"### {fit['method']} — {fit['k']} clusters"))
    table = named_profiles.loc[named_profiles.solution.eq(name),
        ['display_cluster', 'proposed_label', 'papers', 'mean_silhouette',
         'negative_fraction', 'proposal_matched']]
    with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
        display(table.rename(columns={'display_cluster': 'Source cluster',
                                      'proposed_label': 'Proposed topic name'}))
if not named_profiles.proposal_matched.all():
    print('Review the evidence and add membership-matched proposals to:', PROPOSAL_FILE)
else:
    print('All names match the exact source memberships.')
if PROPOSAL_FILE.exists():
    proposals.to_csv(RESULTS / 'label_proposals_snapshot.csv', index=False)


## 4. Inspect the evidence behind a name

Choose a method and one of its original cluster IDs. Read the contrast terms alongside c-TF-IDF, inside/outside coverage, representatives, low-silhouette papers and random examples. The complete membership list is available below the sample; a few representatives cannot establish the scope of every paper.

In [ ]:
INSPECT_SOLUTION = next(name for name in solutions if name.startswith('kmeans_'))  # Or choose a Leiden solution.
INSPECT_CLUSTER = 0  # K-means IDs start at 0; Leiden IDs start at 1.
if INSPECT_SOLUTION not in solutions or INSPECT_CLUSTER not in solutions[INSPECT_SOLUTION]['labels']:
    raise ValueError('Choose an original cluster ID belonging to this exported solution.')
mask = (term_scores.solution == INSPECT_SOLUTION) & (term_scores.cluster == INSPECT_CLUSTER)
display(term_scores.loc[mask, ['ranking', 'rank', 'term', 'inside_count', 'inside_n',
    'outside_count', 'outside_n', 'prevalence_gap', 'score']])
mask = (review_papers.solution == INSPECT_SOLUTION) & (review_papers.cluster == INSPECT_CLUSTER)
with pd.option_context('display.max_colwidth', None, 'display.max_rows', None):
    display(review_papers.loc[mask, ['paper_id', 'title', 'selection', 'cosine_silhouette',
                                   'doi', 'year', 'journal', 'term', 'excerpt_start', 'excerpt_end', 'excerpt', 'abstract']])
    mask = (named_assignments.solution == INSPECT_SOLUTION) & (named_assignments.cluster == INSPECT_CLUSTER)
    # Full membership is saved; show a bounded preview for this larger corpus.
    display(named_assignments.loc[mask, ['paper_id', 'title', 'doi', 'year', 'journal', 'cosine_silhouette']].head(50))
    print(f'{int(mask.sum())} members; full list: {RESULTS / "named_memberships.csv"}')

## 5. Compare the two saved abstract-text partitions

The overlap counts compare the same selected abstracts. ARI and optimal one-to-one matched agreement ignore arbitrary cluster-number differences. They measure agreement between the two methods on one representation, not topic accuracy. This comparison does not relabel or modify either partition.

In [ ]:
agreement, overlap = partition_agreement(
    next(fit['labels'] for fit in solutions.values() if fit['method'] == 'K-means'),
    next(fit['labels'] for fit in solutions.values() if fit['method'] == 'Embedding Leiden'),
)
overlap.index.name = 'K-means source cluster'
overlap.columns = [f'L{cluster}' for cluster in overlap.columns]
overlap.columns.name = 'Embedding Leiden source cluster'
overlap.to_csv(RESULTS / 'method_overlap.csv')
write_json(RESULTS / 'method_agreement.json', agreement)
display(pd.DataFrame([agreement]))
display(overlap)

## 6. Record the interpretation run

The manifest records the source partition snapshot, code and input checksums, vocabulary rules and package versions. Clustering settings and selected seeds come from the source export. Topic names and manual notes remain proposals for the recorded memberships.

In [ ]:
manifest = {
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'workflow': 'interpret_saved_abstract_text_partitions', 'clustering_performed': False,
    'input_id': INPUT_ID, 'corpus_size': len(papers), 'input_hashes': provenance['input_hashes'],
    'partition_manifest_sha256': file_hash(PARTITION_DIR / 'partition_manifest.json'),
    'source_assignments_sha256': partition_manifest['assignments_sha256'],
    'source_partitions': partition_manifest['solutions'],
    'config': {'text': 'title + blank line + abstract', 'min_df': 3, 'max_df': .9,
               'ngrams': [1, 2], 'minimum_cluster_term_support': 'max(2, ceil(0.1*n))',
               'review_random_seed': 42},
    'naming': 'Recorded AI-assisted analyst proposals; not automatic naming or user validation',
    'label_proposals_sha256': file_hash(PROPOSAL_FILE) if PROPOSAL_FILE.exists() else None,
    'all_names_match_memberships': bool(named_profiles.proposal_matched.all()),
    'python': sys.version,
    'packages': {p: version(p) for p in ['numpy', 'pandas', 'scipy', 'scikit-learn']},
    'code_sha256': {p.name: file_hash(p) for p in [ROOT / 'src/batill/abstract_topic_analysis.py',
        ROOT / 'src/batill/abstract_clustering.py', ROOT / 'src/batill/topic_analysis.py']},
    'artifacts_sha256': {p.name: file_hash(p) for p in RESULTS.iterdir()
                        if p.is_file() and p.name != 'run_manifest.json'},
}
write_json(RESULTS / 'run_manifest.json', manifest)
print('Saved abstract topic evidence and names:', RESULTS)


## Reproduce and share the complete analysis

The supervisor handoff is `outputs/topic-analysis-reproduction.tar.gz`. It contains the exact saved inputs, memberships, naming proposals, code, pinned dependency lock and reference evidence for all three topic notebooks. Sending this notebook alone is insufficient.

After unpacking, use Python 3.12 inside the bundle folder:

```bash
python3.12 -m venv .venv
.venv/bin/python -m pip install -r configs/topic-analysis-requirements.lock
.venv/bin/python scripts/reproduce_topic_analyses.py --check
```

The checker runs each notebook in a fresh process with clustering, extraction, model loading and network calls disabled. It requires exact checksums for the deterministic tables, names, excerpts and review cards. Timestamps, absolute paths in run manifests and notebook display output are excluded from byte comparison. Names are recorded interpretive inputs; reusing their CSV reproduces the names, while rerunning recomputes the supporting evidence. See the README section **Reproduce all three saved-partition topic analyses** for scope, environment limits and update instructions.
